# 12 · Klassen und Objekte

**Lernziele**

- Du definierst Klassen mit Attributen und Methoden.
- Du erzeugst Objekte und arbeitest mit ihnen.
- Du gibst Objekte lesbar aus und vergleichst sie.
- Du nutzt `@dataclass` für reine Datenklassen.
- Du legst feste Auswahlwerte als `Enum` an.
- Du erweiterst Klassen durch Vererbung und überschreibst Methoden.

> Grundlage: Inden, *Einstieg in Python*, Kapitel 4 und 6.

## Warum Klassen?

Bisher hast du Daten (z. B. Dictionaries) und Funktionen getrennt. Eine **Klasse** bündelt beides: Sie beschreibt, welche Daten ein Ding hat (**Attribute**) und was es kann (**Methoden**). Ein **Objekt** ist ein konkretes Exemplar einer Klasse.

| Klasse (Bauplan) | Objekte (Exemplare) |
| --- | --- |
| `Car` | Toms Audi, Jims VW |
| `BankAccount` | Konto 4711, Konto 0815 |

## Eine erste Klasse

`__init__` wird beim Erzeugen aufgerufen und legt die Attribute an. `self` ist das Objekt selbst.

In [ ]:
class Car:
    def __init__(self, brand, color, horse_power):
        self.brand = brand
        self.color = color
        self.horse_power = horse_power

    def paint(self, new_color):
        self.color = new_color

    def apply_tuning_kit(self):
        self.horse_power += 150


my_car = Car("VW", "orange", 123)
print(my_car.brand, my_car.color, my_car.horse_power)

my_car.paint("green")
my_car.apply_tuning_kit()
print(my_car.color, my_car.horse_power)

> Für Java- und C#-Umsteiger: Es gibt kein `new`, kein `this` (dafür explizit `self`), und Attribute werden nicht vorab deklariert.

**Kurzübung 1 (self vergessen):** `Contact("Ada Lovelace", "ada@example.com").signature()` soll `Ada Lovelace <ada@example.com>` liefern. Im Code stecken zwei Fehler. Finde und behebe beide.

In [ ]:
class Contact:
    def __init__(self, name: str, email: str) -> None:
        name = name
        email = email

    def signature() -> str:
        return f"{self.name} <{self.email}>"


ada = Contact("Ada Lovelace", "ada@example.com")
print(ada.signature())

In [ ]:
class Contact:
    def __init__(self, name: str, email: str) -> None:
        self.name = name  # ohne self. bleibt der Wert nur lokal in __init__
        self.email = email

    def signature(self) -> str:  # jede Methode bekommt self als ersten Parameter
        return f"{self.name} <{self.email}>"


ada = Contact("Ada Lovelace", "ada@example.com")
print(ada.signature())
# Fehler 1: def signature(): ohne self -> TypeError: Contact.signature() takes
#           0 positional arguments but 1 was given
# Fehler 2: name = name ohne self. -> AttributeError beim Lesen von self.name

## Lesbare Ausgabe mit `__str__`

Ohne Hilfe sieht ein Objekt in der Ausgabe kryptisch aus:

In [ ]:
print(my_car)

Methoden mit doppelten Unterstrichen (**Dunder-Methoden**) haben eine besondere Bedeutung. `__str__` legt fest, was `print()` zeigt:

In [ ]:
class Car:
    def __init__(self, brand, color, horse_power):
        self.brand = brand
        self.color = color
        self.horse_power = horse_power

    def __str__(self):
        return f"{self.brand} ({self.color}, {self.horse_power} PS)"


print(Car("Audi", "blue", 275))

**Kurzübung 2 (Rechnung lesbar ausgeben):** Ergänze in `Invoice` eine Methode `__str__`, sodass `print(invoice)` den Text `Rechnung 2026-117: 249.90 €` zeigt.

Untersuchen: Was passiert, wenn `__str__` den Text mit `print()` ausgibt, statt ihn mit `return` zurückzugeben?

In [ ]:
class Invoice:
    def __init__(self, number: str, amount: float) -> None:
        self.number = number
        self.amount = amount


invoice = Invoice("2026-117", 249.9)
print(invoice)

In [ ]:
class Invoice:
    def __init__(self, number: str, amount: float) -> None:
        self.number = number
        self.amount = amount

    def __str__(self) -> str:
        return f"Rechnung {self.number}: {self.amount:.2f} €"


invoice = Invoice("2026-117", 249.9)
print(invoice)

# Untersuchen: Gibt __str__ mit print() aus und liefert nichts zurück, erscheint
# der Text zwar, danach folgt aber TypeError: Invoice.__str__() must return
# a str, not NoneType. __str__ muss den Text zurückgeben.

## Objekte vergleichen

`==` vergleicht standardmäßig, ob es **dasselbe** Objekt ist, nicht ob die Inhalte gleich sind:

In [ ]:
toms_car = Car("Audi", "blue", 275)
jims_car = Car("Audi", "blue", 275)
print(toms_car == jims_car)

**Kurzübung 3 (Zwei gleiche Kontakte):** Welche Ausgabe erzeugen `Contact("Ada", "ada@example.com") == Contact("Ada", "ada@example.com")` und `print([ada])`, und warum? Ergänze `__eq__` und `__repr__`, sodass der Vergleich `True` liefert und die Liste lesbar ist.

- Untersuchen: Was liefert `ada is Contact("Ada", "ada@example.com")`, nachdem `__eq__` da ist?
- Ändern: `__eq__` soll E-Mail-Adressen ohne Groß-/Kleinschreibung vergleichen.

In [ ]:
class Contact:
    def __init__(self, name: str, email: str) -> None:
        self.name = name
        self.email = email


ada = Contact("Ada", "ada@example.com")
print(Contact("Ada", "ada@example.com") == Contact("Ada", "ada@example.com"))
print([ada])

In [ ]:
# Vorhersage: False, denn ohne __eq__ prüft == nur, ob es dasselbe Objekt ist.
# print([ada]) zeigt <__main__.Contact object at 0x...>: Listen nutzen __repr__,
# nicht __str__.


class Contact:
    def __init__(self, name: str, email: str) -> None:
        self.name = name
        self.email = email

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Contact):
            return NotImplemented
        # Ändern: E-Mail ohne Groß-/Kleinschreibung vergleichen
        return (self.name, self.email.casefold()) == (
            other.name,
            other.email.casefold(),
        )

    def __repr__(self) -> str:
        return f"Contact({self.name!r}, {self.email!r})"


ada = Contact("Ada", "ada@example.com")
print(ada == Contact("Ada", "ADA@example.com"))  # True
print(ada is Contact("Ada", "ada@example.com"))  # False: is prüft Identität
print([ada])  # [Contact('Ada', 'ada@example.com')]

`__eq__` selbst zu schreiben ist Fleißarbeit. Einfacher geht es mit einer **Datenklasse**.

## Datenklassen mit `@dataclass`

Für Klassen, die vor allem Daten halten, erzeugt `@dataclass` automatisch `__init__`, eine lesbare Ausgabe und `==`. Die Attribute werden mit Type Hints angegeben.

In [ ]:
from dataclasses import dataclass


@dataclass
class Car:
    brand: str
    color: str
    horse_power: int

    def apply_tuning_kit(self) -> None:
        self.horse_power += 150


toms_car = Car("Audi", "blue", 275)
jims_car = Car("Audi", "blue", 275)
print(toms_car)
print(toms_car == jims_car)

> Faustregel: Für reine Daten `@dataclass`, für Objekte mit viel Verhalten eine normale Klasse. Beides sind ganz normale Klassen.

**Kurzübung 4 (Alle Teams teilen eine Liste):** Ada kommt ins Team Vertrieb, steht danach aber auch im Team Support. Finde den Fehler und behebe ihn mit einer Datenklasse. Was passiert, wenn du dort `members: list[str] = []` schreibst? Die Fehlermeldung nennt den Ausweg: `field(default_factory=list)` aus dem Modul `dataclasses`.

In [ ]:
class Team:
    members = []

    def __init__(self, name: str) -> None:
        self.name = name


sales, support = Team("Vertrieb"), Team("Support")
sales.members.append("Ada")
print(support.members)

In [ ]:
from dataclasses import field


@dataclass
class Team:
    name: str
    members: list[str] = field(default_factory=list)  # jede Instanz: eigene Liste


sales, support = Team("Vertrieb"), Team("Support")
sales.members.append("Ada")
print(support.members)  # []

# Fehler: members = [] war ein Klassenattribut, das alle Teams gemeinsam haben.
# members: list[str] = [] lehnt @dataclass ab: ValueError "mutable default
# <class 'list'> for field members is not allowed: use default_factory".
# Dieselbe Falle wie bei veränderlichen Standardwerten von Funktionen (07).

## Interne Attribute

Python kennt kein `private`. Die **Konvention**: Ein Unterstrich am Anfang bedeutet „intern, bitte nicht von außen ändern“. Der Zugriff erfolgt über Methoden.

In [ ]:
class Counter:
    def __init__(self):
        self._value = 0

    @property
    def value(self) -> int:
        return self._value

    def increment(self) -> None:
        self._value += 1

    def reset(self) -> None:
        self._value = 0


counter = Counter()
counter.increment()
counter.increment()
print(counter.value)
counter.reset()
print(counter.value)

`@property` macht `value` von außen **lesbar wie ein Attribut**, aber nicht schreibbar:

In [ ]:
counter.value = 100

**Kurzübung 5 (Wartenummern im Bürgerbüro):** Schreibe eine Klasse `TicketMachine`, die die zuletzt vergebene Nummer intern in `_last_number` speichert (Start: 0). Die Methode `draw()` erhöht sie um 1 und gibt die neue Nummer zurück, die Property `last_number` macht sie von außen lesbar. Ziehe drei Nummern und gib danach `last_number` aus. Was passiert bei `machine.last_number = 0`?

In [ ]:
class TicketMachine:
    def __init__(self) -> None:
        self._last_number = 0  # intern: nur die Klasse selbst ändert den Wert

    @property
    def last_number(self) -> int:
        return self._last_number

    def draw(self) -> int:
        self._last_number += 1
        return self._last_number


machine = TicketMachine()
for _ in range(3):
    print("Ihre Nummer:", machine.draw())
print("Zuletzt vergeben:", machine.last_number)  # 3

try:
    machine.last_number = 0  # Property ohne Setter: nur lesbar
except AttributeError as error:
    print("Nicht erlaubt:", error)

## Aufzählungen mit `Enum`

Manche Werte haben nur wenige erlaubte Möglichkeiten: Ampelfarben, Spielkartenfarben, der Status einer Bestellung. Als Strings wie `"rot"` sind sie anfällig für Tippfehler: `"Rot"` oder `"rto"` fällt erst spät oder gar nicht auf. Eine **Aufzählung** (*Enumeration*) legt die erlaubten Werte als Klasse fest. Die Mitglieder schreibt man in Großbuchstaben, weil es Konstanten sind:

In [ ]:
from enum import Enum, StrEnum, auto


class TrafficLight(Enum):
    RED = 1
    YELLOW = 2
    GREEN = 3


light = TrafficLight.RED
print(light)
print(light.name, light.value)
print(light == TrafficLight.RED)

Jedes Mitglied hat einen Namen (`name`) und einen Wert (`value`). Verglichen wird mit `==` oder `is`. Ein Tippfehler fällt sofort auf, statt still falsch zu laufen:

In [ ]:
TrafficLight.RDE

Über eine Enum kannst du iterieren, in der Reihenfolge der Definition:

In [ ]:
for color in TrafficLight:
    print(color.name, color.value)

print(TrafficLight(2))  # aus einem gespeicherten Wert wird wieder ein Mitglied

Sind dir die konkreten Werte egal, vergibt `auto()` sie. `StrEnum` verhält sich zusätzlich wie ein String und bekommt mit `auto()` den Namen in Kleinbuchstaben als Wert. Das ist praktisch, wenn die Werte in Dateien, JSON oder Ausgaben landen:

In [ ]:
class OrderStatus(StrEnum):
    OPEN = auto()  # Wert: "open"
    SHIPPED = auto()
    DELIVERED = auto()


status = OrderStatus.SHIPPED
print(status, status == "shipped")
print(f"Status: {status}")

Enums und `match` (Notebook 03) passen gut zusammen. Weil `TrafficLight.RED` einen Punkt enthält, **vergleicht** `case` mit diesem Wert, statt ihn einzufangen:

In [ ]:
def next_light(light: TrafficLight) -> TrafficLight:
    match light:  # vereinfacht, ohne Rot-Gelb
        case TrafficLight.RED:
            return TrafficLight.GREEN
        case TrafficLight.GREEN:
            return TrafficLight.YELLOW
        case TrafficLight.YELLOW:
            return TrafficLight.RED


light = TrafficLight.RED
for _ in range(4):
    print(light.name)
    light = next_light(light)

**Kurzübung 6 (Kleidergrößen als Enum):** Ein Shop sortiert Größen als Text: `sorted(["XL", "S", "L", "M"])`. Welches Ergebnis entsteht? Schreibe eine Enum `Size` und sortiere nach ihrer Reihenfolge. Was passiert bei `Size.XXL` im Vergleich zum Text `"XXL"`?

- Untersuchen: Was liefern `Size.S.name` und `Size.S.value`?
- Ändern: Ergänze `XXL` so, dass sie hinter `XL` einsortiert wird.

In [ ]:
print(sorted(["XL", "S", "L", "M"]))

In [ ]:
class Size(Enum):
    S = "S"
    M = "M"
    L = "L"
    XL = "XL"
    XXL = "XXL"  # Ändern: hinten anfügen, Reihenfolge = Definitionsreihenfolge


print(sorted(["XL", "S", "L", "M"]))  # ['L', 'M', 'S', 'XL']: als Text falsch
print(sorted([Size.XXL, Size.S, Size.L, Size.M], key=list(Size).index))
print(Size.S.name, Size.S.value, Size["M"], Size("XL"))  # Untersuchen
# Ohne XXL in der Enum löst Size.XXL sofort einen AttributeError aus,
# der Text "XXL" dagegen wird ohne Meldung mitsortiert und verglichen.

In [ ]:
Size.XXXL  # Tippfehler: AttributeError sofort, der Text "XXXL" bliebe unbemerkt

## Vererbung

Eine Klasse kann von einer anderen **erben**: Sie übernimmt alles und kann Methoden ergänzen oder **überschreiben**. Mit `super()` rufst du die Version der Elternklasse auf.

Beispiel: Ein Punktezähler, der bei 100 überläuft und dafür ein Bonusleben vergibt.

In [ ]:
class CounterWithOverflow(Counter):
    MAX_VALUE = 100

    def __init__(self):
        super().__init__()
        self.overflow_count = 0

    def increment(self) -> None:
        super().increment()
        if self.value >= self.MAX_VALUE:
            self._value = 0
            self.overflow_count += 1


points = CounterWithOverflow()
for _ in range(2026):
    points.increment()

print(f"Punkte: {points.value}, Bonusleben: {points.overflow_count}")
print(isinstance(points, Counter))  # ein CounterWithOverflow ist auch ein Counter

Unterschiedliche Klassen können dieselbe Methode unterschiedlich umsetzen. Der aufrufende Code muss den genauen Typ nicht kennen (**Polymorphie**):

In [ ]:
class Shape:
    def area(self) -> float:
        raise NotImplementedError


class Rectangle(Shape):
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height


class Circle(Shape):
    def __init__(self, radius: float):
        self.radius = radius

    def area(self) -> float:
        return 3.14159 * self.radius ** 2


for shape in [Rectangle(3, 4), Circle(1)]:
    print(f"{type(shape).__name__}: {shape.area():.2f}")

> **Vererbung sparsam einsetzen.** Sie passt, wenn gilt „X **ist ein** Y“. Oft ist es einfacher, ein Objekt als Attribut in einem anderen zu verwenden („X **hat ein** Y“).

**Kurzübung 7 (Besprechungsraum voll):** Leite von `Counter` eine Klasse `RoomCounter` ab, die Personen in einem Besprechungsraum mit 8 Plätzen zählt. Lege die Grenze als Klassenattribut `MAX_VALUE = 8` an und überschreibe `increment()`: Ist der Raum voll, gibt die Methode `Raum voll` aus, sonst zählt sie mit `super().increment()` weiter. Rufe `increment()` zehnmal auf. Welche Ausgabe erwartest du, und welchen Wert hat `value` danach?

In [ ]:
class RoomCounter(Counter):
    MAX_VALUE = 8

    def increment(self) -> None:
        if self.value >= self.MAX_VALUE:
            print("Raum voll")
        else:
            super().increment()  # das eigentliche Zählen erledigt Counter


room = RoomCounter()
for _ in range(10):
    room.increment()
print(room.value)  # 8, davor zweimal "Raum voll"

## Übungen

**Übung 1 (Produkt als Klasse, leicht):** Schreibe eine Klasse `Product` mit den Attributen `name`, `net_price` und `vat_rate` (Mehrwertsteuersatz, Standardwert `0.19`). Die Methode `price_with_vat()` liefert den Bruttopreis. Lege zwei Produkte an: Brot für 2,80 € netto mit 7 % und Kopfhörer für 49,90 € netto mit dem Standardsatz. Gib beide mit Namen und Bruttopreis auf zwei Nachkommastellen aus.

Untersuchen: Was davon ist die Klasse, eine Instanz, eine Methode und das Ergebnis eines Methodenaufrufs: `Product`, `Product("Brot", 2.5)`, `p.price_with_vat` und `p.price_with_vat()` (mit `p = Product("Brot", 2.5)`)? Prüfe deine Antwort mit `print()` und `type()`.

In [ ]:
class Product:
    def __init__(self, name: str, net_price: float, vat_rate: float = 0.19) -> None:
        self.name = name
        self.net_price = net_price
        self.vat_rate = vat_rate

    def price_with_vat(self) -> float:
        return self.net_price * (1 + self.vat_rate)


bread = Product("Brot", 2.80, 0.07)
headphones = Product("Kopfhörer", 49.90)
for product in [bread, headphones]:
    print(f"{product.name}: {product.price_with_vat():.2f} €")

# Untersuchen
p = Product("Brot", 2.5)
print(Product)  # <class '__main__.Product'>: die Klasse
print(p)  # <__main__.Product object at 0x...>: eine Instanz
print(p.price_with_vat)  # <bound method Product.price_with_vat of ...>: Methode
print(type(p.price_with_vat()))  # <class 'float'>: Ergebnis des Aufrufs

**Übung 2 (Rechnungspositionen, leicht):** Schreibe eine Datenklasse `LineItem` mit `description`, `quantity` und `unit_price` und einer Methode `total()`, die den Preis der Position liefert. Lege die Positionen einer Büromaterial-Bestellung an: 5 Pakete Druckerpapier zu 4,99 €, 2 Toner zu 39,90 € und 10 Ordner zu 2,49 €. Gib jede Position als Zeile aus und darunter die Rechnungssumme (129,65 €): erst mit einer eigenen Schleife, dann mit `sum()`. Welche Fassung nimmst du, und warum?

Untersuchen: Was liefert `LineItem("Ordner", 10, 2.49) == LineItem("Ordner", 10, 2.49)`, und warum anders als beim `Contact` aus Kurzübung 3, bevor du dort `__eq__` ergänzt hast?

In [ ]:
from dataclasses import dataclass


@dataclass
class LineItem:
    description: str
    quantity: int
    unit_price: float

    def total(self) -> float:
        return self.quantity * self.unit_price


order = [
    LineItem("Druckerpapier", 5, 4.99),
    LineItem("Toner", 2, 39.90),
    LineItem("Ordner", 10, 2.49),
]
for item in order:
    print(
        f"{item.quantity:>3} x {item.description:<14}"
        f"{item.unit_price:>7.2f} € {item.total():>8.2f} €"
    )

# Fassung 1: eigene Schleife
invoice_total = 0.0
for item in order:
    invoice_total += item.total()
print(f"Summe: {invoice_total:.2f} €")

# Fassung 2: sum() mit Generator-Ausdruck, kürzer und ohne Hilfsvariable
print(f"Summe: {sum(item.total() for item in order):.2f} €")

# Untersuchen: True. @dataclass erzeugt __eq__, das alle Felder vergleicht;
# Contact ohne eigenes __eq__ verglich nur, ob es dasselbe Objekt ist.
print(LineItem("Ordner", 10, 2.49) == LineItem("Ordner", 10, 2.49))

**Übung 3 (Versandart, leicht):** Im Webshop wählt die Kundschaft eine Versandart, das Formular liefert sie als Text: `"standard"`, `"express"` oder `"pickup"`. Lege eine `StrEnum` `Shipping` mit `auto()` an und schreibe eine Funktion `shipping_cost(method)`, die mit `match` die Kosten liefert: Standard 4,90 €, Express 9,90 €, Abholung kostenlos. Wandle die Texte im Startcode in `Shipping`-Mitglieder um und berechne die Versandkosten aller Bestellungen (zusammen 19,70 €).

Untersuchen: Was passiert bei `Shipping("Express")` mit großem E?

In [ ]:
# Versandart je Bestellung, so wie das Formular sie liefert
orders = ["standard", "express", "pickup", "standard"]

In [ ]:
from enum import StrEnum, auto


class Shipping(StrEnum):
    STANDARD = auto()  # Wert: "standard"
    EXPRESS = auto()
    PICKUP = auto()


def shipping_cost(method: Shipping) -> float:
    match method:
        case Shipping.STANDARD:
            return 4.90
        case Shipping.EXPRESS:
            return 9.90
        case Shipping.PICKUP:
            return 0.0
        case _:  # fällt auf, wenn später eine Versandart dazukommt
            raise ValueError(f"Unbekannte Versandart: {method}")


total = 0.0
for text in orders:
    method = Shipping(text)  # aus dem Text wird ein Mitglied
    cost = shipping_cost(method)
    print(f"{method:<10} {cost:>5.2f} €")
    total += cost
print(f"Summe: {total:.2f} €")  # 19.70 €

# Untersuchen: Die Werte sind klein geschrieben, "Express" gibt es nicht.
try:
    Shipping("Express")
except ValueError as error:
    print(error)  # 'Express' is not a valid Shipping

**Übung 4 (Parkticket, mittel):** Schreibe eine Klasse `ParkingTicket` mit Kennzeichen und Einfahrtszeit. Die Methode `price(leaving)` rechnet nach der Regel aus 03 (angefangene Stunde 2 €, höchstens 15 €), `print(ticket)` zeigt `Ticket K-AB 123, Einfahrt 16.11. 08:05`. Lege Stundenpreis und Höchstsatz als Klassenattribute an.

In [ ]:
from datetime import datetime


class ParkingTicket:
    RATE_PER_HOUR = 2  # Klassenattribute: gelten für alle Tickets
    DAILY_MAX = 15

    def __init__(self, plate: str, entry: datetime) -> None:
        self.plate = plate  # Instanzattribute: je Ticket verschieden
        self.entry = entry

    def price(self, leaving: datetime) -> int:
        minutes = int((leaving - self.entry).total_seconds() // 60)
        hours = minutes // 60 + (1 if minutes % 60 else 0)  # angefangene Stunde
        return min(hours * self.RATE_PER_HOUR, self.DAILY_MAX)

    def __str__(self) -> str:  # gibt Text zurück, druckt nicht selbst
        return f"Ticket {self.plate}, Einfahrt {self.entry:%d.%m. %H:%M}"


ticket = ParkingTicket("K-AB 123", datetime(2026, 11, 16, 8, 5))
print(ticket)
print(ticket.price(datetime(2026, 11, 16, 10, 20)), "€")  # 135 Minuten -> 6 €

**Übung 5 (Terminkalender: Klasse oder Datenklasse?, mittel):** Der Startcode enthält zwei korrekte Fassungen eines Termins, wie sie auch eine KI vorschlagen könnte: als normale Klasse und als Datenklasse. Sage vorher, was die drei `print` ausgeben, und entscheide: Welche nimmst du, und warum? Ergänze dann in deiner Wahl eine Property `end` und eine Methode `overlaps(other)` und finde in `day` alle Paare, die sich überschneiden.

- Untersuchen: Wie viele Zeilen braucht jede Fassung? Was musst du bei A ändern, wenn ein Feld `room` dazukommt, was bei B?
- Ändern: Ergänze in der gewählten Fassung eine Property `end` und eine Methode `overlaps(other)`.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta


class AppointmentA:
    def __init__(self, title: str, start: datetime, duration: timedelta) -> None:
        self.title = title
        self.start = start
        self.duration = duration

    def __repr__(self) -> str:
        return f"AppointmentA({self.title!r}, {self.start!r}, {self.duration!r})"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, AppointmentA):
            return NotImplemented
        return (self.title, self.start, self.duration) == (
            other.title,
            other.start,
            other.duration,
        )


@dataclass
class AppointmentB:
    title: str
    start: datetime
    duration: timedelta


a = AppointmentA("Kunde", datetime(2026, 11, 16, 9, 0), timedelta(hours=1))
b = AppointmentB("Kunde", datetime(2026, 11, 16, 9, 0), timedelta(hours=1))
print(a == AppointmentA("Kunde", datetime(2026, 11, 16, 9, 0), timedelta(hours=1)))
print(b == AppointmentB("Kunde", datetime(2026, 11, 16, 9, 0), timedelta(hours=1)))
print(b)

# Termine eines Tages als (Titel, Beginn, Dauer)
day = [
    ("Standup", datetime(2026, 11, 16, 9, 30), timedelta(minutes=15)),
    ("Kunde", datetime(2026, 11, 16, 9, 0), timedelta(hours=1)),
    ("Review", datetime(2026, 11, 16, 10, 0), timedelta(minutes=30)),
]

In [ ]:
# Vorhersage: True, True und
# AppointmentB(title='Kunde', start=datetime.datetime(2026, 11, 16, 9, 0),
#              duration=datetime.timedelta(seconds=3600))
# Untersuchen: A braucht 15 Zeilen (ohne Leerzeilen), B nur 5. Kommt ein Feld
# room dazu, änderst du bei A __init__, __repr__ und __eq__, bei B eine Zeile.


@dataclass
class Appointment:  # Wahl: Datenklasse, weniger Code und neue Felder kosten eine Zeile
    title: str
    start: datetime
    duration: timedelta

    @property
    def end(self) -> datetime:
        return self.start + self.duration

    def overlaps(self, other: Appointment) -> bool:
        # < statt <=: endet ein Termin um 10:00, darf der nächste um 10:00 beginnen
        return self.start < other.end and other.start < self.end


appointments = sorted(
    (Appointment(*entry) for entry in day), key=lambda appointment: appointment.start
)
for i, first in enumerate(appointments):
    for second in appointments[i + 1 :]:
        if first.overlaps(second):
            print(f"Konflikt: {first.title} / {second.title}")

**Übung 6 (Girokonto mit Dispo, mittel):** Die Basisklasse `BankAccount` steht in der Aufgabenzelle; `withdraw` prüft gegen `available()`. Leite `CheckingAccount` mit `overdraft_limit` (Standard 500 €) ab, rufe in `__init__` `super().__init__` auf und überschreibe nur `available()`: Kontostand plus Dispo. Prüfe, dass das geerbte `withdraw` die neue Grenze automatisch nutzt.

In [ ]:
class InsufficientFundsError(ValueError):
    pass


class BankAccount:
    def __init__(self, owner: str, balance: float = 0) -> None:
        self.owner = owner
        self.balance = balance

    def available(self) -> float:
        return self.balance

    def withdraw(self, amount: float) -> None:
        if amount > self.available():
            raise InsufficientFundsError(f"Verfügbar sind nur {self.available():.2f} €")
        self.balance -= amount

In [ ]:
class CheckingAccount(BankAccount):
    def __init__(
        self, owner: str, balance: float = 0, overdraft_limit: float = 500
    ) -> None:
        super().__init__(owner, balance)  # legt owner und balance an
        self.overdraft_limit = overdraft_limit

    def available(self) -> float:  # nur die Prüfgröße überschreiben
        return super().available() + self.overdraft_limit


giro = CheckingAccount("Grace", 100)
giro.withdraw(400)  # geerbtes withdraw ruft das neue available() auf
print(giro.balance)  # -300
try:
    giro.withdraw(300)
except InsufficientFundsError as error:
    print("Abgelehnt:", error)

**Übung 7 (Konto mit Buchungen, schwer):** Das Konto speichert jede Buchung als `Booking(timestamp, amount, text)`, der Kontostand ist die Summe der Buchungen. Überziehung löst `InsufficientFundsError` aus 09 aus, `statement(year, month)` liefert die Buchungen eines Monats. Geld in `Decimal`.

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from decimal import Decimal


class InsufficientFundsError(ValueError):
    pass


@dataclass
class Booking:
    timestamp: datetime
    amount: Decimal
    text: str


@dataclass
class BankAccount:
    owner: str
    bookings: list[Booking] = field(default_factory=list)

    @property
    def balance(self) -> Decimal:
        # berechnet statt mitgeführt: kann nie von den Buchungen abweichen
        return sum((booking.amount for booking in self.bookings), Decimal(0))

    def deposit(self, amount: Decimal, text: str, when: datetime) -> None:
        if amount <= 0:
            raise ValueError("Betrag muss positiv sein")
        self.bookings.append(Booking(when, amount, text))

    def withdraw(self, amount: Decimal, text: str, when: datetime) -> None:
        if amount > self.balance:
            raise InsufficientFundsError(f"Es fehlen {amount - self.balance} €")
        self.bookings.append(Booking(when, -amount, text))

    def statement(self, year: int, month: int) -> list[Booking]:
        return [
            booking
            for booking in self.bookings
            if (booking.timestamp.year, booking.timestamp.month) == (year, month)
        ]


account = BankAccount("Ada")
account.deposit(Decimal("2500.00"), "Gehalt", datetime(2026, 10, 30))
account.withdraw(Decimal("850.00"), "Miete", datetime(2026, 11, 1))
print(account.balance, [booking.text for booking in account.statement(2026, 11)])
try:
    account.withdraw(Decimal("2000.00"), "Urlaub", datetime(2026, 11, 2))
except InsufficientFundsError as error:
    print("Abgelehnt:", error)

**Übung 8 (Bestellstatus, schwer):** Eine Bestellung durchläuft `OrderStatus` (offen, bezahlt, versandt, zugestellt, storniert). `advance(new_status, when)` erlaubt nur gültige Übergänge und löst sonst einen `ValueError` aus; jeder Wechsel wird mit Zeitstempel gespeichert. Versuche, eine versandte Bestellung zu stornieren.

Variante Prompt-Problem (optional): Die Tests unten sind vorgegeben. Schreibe nur den Prompt für deine Chat-KI, kopiere die Antwort ins Notebook und verbessere den Prompt, bis die Tests grün sind. Danach: Erkläre den erzeugten Code in eigenen Worten und sage vorher, was `Order("B-2").advance(OrderStatus.DELIVERED, ...)` tut. Ohne KI-Zugang: die typische KI-Antwort aus der Lösung prüfen.

In [ ]:
# Typische KI-Antwort auf einen Prompt, der Status, erlaubte Übergänge,
# ValueError und Historie nennt; zugleich Lösung ohne KI
from dataclasses import dataclass, field
from datetime import datetime
from enum import StrEnum


class OrderStatus(StrEnum):
    OPEN = "offen"
    PAID = "bezahlt"
    SHIPPED = "versandt"
    DELIVERED = "zugestellt"
    CANCELLED = "storniert"


ALLOWED = {
    OrderStatus.OPEN: {OrderStatus.PAID, OrderStatus.CANCELLED},
    OrderStatus.PAID: {OrderStatus.SHIPPED, OrderStatus.CANCELLED},
    OrderStatus.SHIPPED: {OrderStatus.DELIVERED},
}


@dataclass
class Order:
    number: str
    status: OrderStatus = OrderStatus.OPEN
    history: list[tuple[datetime, OrderStatus]] = field(default_factory=list)

    def advance(self, new_status: OrderStatus, when: datetime) -> None:
        if new_status not in ALLOWED.get(self.status, set()):
            raise ValueError(f"{self.status} -> {new_status} ist nicht erlaubt")
        self.status = new_status
        self.history.append((when, new_status))


order = Order("A-1001")
order.advance(OrderStatus.PAID, datetime(2026, 11, 2, 10, 5))
order.advance(OrderStatus.SHIPPED, datetime(2026, 11, 3, 8, 0))
try:
    order.advance(OrderStatus.CANCELLED, datetime(2026, 11, 3, 9, 0))
except ValueError as error:
    print(error)

# Vorhersage: Order("B-2").advance(OrderStatus.DELIVERED, ...) löst
# ValueError "offen -> zugestellt ist nicht erlaubt" aus, denn von "offen"
# geht es nur nach "bezahlt" oder "storniert".

In [ ]:
# Vorgegebene Tests für die Variante Prompt-Problem
# (laufen erst mit Order und OrderStatus aus der Zelle darüber)
from datetime import datetime

order = Order("A-1001")
assert order.status == OrderStatus.OPEN and order.history == []
order.advance(OrderStatus.PAID, datetime(2026, 11, 2, 10, 5))
order.advance(OrderStatus.SHIPPED, datetime(2026, 11, 3, 8, 0))
assert order.status == OrderStatus.SHIPPED and len(order.history) == 2
try:
    order.advance(OrderStatus.CANCELLED, datetime(2026, 11, 3, 9, 0))
    raise AssertionError("versandt -> storniert darf nicht gehen")
except ValueError:
    pass
# jede Bestellung hat ihre eigene Historie
assert Order("A-1002").history is not Order("A-1003").history
print("Alle Tests bestanden")

## Zusammenfassung

- `class Name:` mit `__init__(self, ...)` legt Attribute an, Methoden bekommen `self`.
- `__str__` steuert die Ausgabe mit `print()`.
- `@dataclass` erzeugt `__init__`, Ausgabe und `==` automatisch.
- `_name` heißt „intern“; `@property` macht Werte lesbar, aber nicht schreibbar.
- `Enum` für feste Auswahlwerte, `StrEnum` wenn sie sich wie Strings verhalten sollen; passt zu `match`.
- Vererbung: `class Child(Parent):`, `super().methode()` ruft die Elternversion auf.